# Quantum gravity: scales, effective theories, and evidence

Evidence-led calculations; explanatory companion: [quantum_gravity.md](../premium/quantum_gravity.md). Run from the project root or this notebook directory. Plotted curves are theoretical illustrations, never quantum-gravity observations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.constants import G, c, hbar, k_B, M_sun

# No external files/data are required. Resolve module paths for reproducible context.
HERE = Path.cwd()
PROJECT = HERE if (HERE / "notebooks").is_dir() and (HERE / "premium").is_dir() else HERE.parent
print("Project root:", PROJECT.resolve())
print("Inputs below are constants, algebraic definitions, or explicitly chosen toy parameters; no observational catalog is loaded.")

## 1. Dimensional construction of Planck scales

Measured inputs: $G$, $c$, $\hbar$, $k_B$ from Astropy/CODATA (the exactness/uncertainty status of constants follows SI/CODATA). Derivation: assume $\ell=G^a\hbar^bc^d$; matching mass, length, and time dimensions gives $-a+b=0$, $3a+2b+d=1$, and $-2a-b-d=0$. Thus $b=a$, $d=-3a$, $2a=1$, so $\ell_P=\sqrt{\hbar G/c^3}$. Then $t_P=\ell_P/c$, $m_P=\sqrt{\hbar c/G}$, $E_P=m_Pc^2$, and $T_P=E_P/k_B$. These are derived scales, not observed objects.

In [ ]:
# Symbolic verification of dimensions and identities; exponents are (M,L,T).
Gdim = sp.Matrix([-1, 3, -2]); hdim = sp.Matrix([1, 2, -1]); cdim = sp.Matrix([0, 1, -1])
a,b,d = sp.symbols('a b d')
sol = sp.solve([sp.Eq(-a+b,0), sp.Eq(3*a+2*b+d,1), sp.Eq(-2*a-b-d,0)], [a,b,d])
assert sol == {a: sp.Rational(1,2), b: sp.Rational(1,2), d: sp.Rational(-3,2)}
assert sp.simplify((Gdim/2+hdim/2-3*cdim/2) - sp.Matrix([0,1,0])) == sp.zeros(3,1)
Gv, hv, cv, kv = G.si, hbar.si, c.si, k_B.si
lP = np.sqrt((hbar*G/c**3).to(u.m**2)).to(u.m)
tP = (lP/c).to(u.s)
mP = np.sqrt((hbar*c/G).to(u.kg**2)).to(u.kg)
EP = (mP*c**2).to(u.J)
TP = (EP/k_B).to(u.K)
planck = pd.DataFrame({'derived scale':['length','time','mass','energy','temperature'], 'value':[lP.value,tP.value,mP.value,EP.value,TP.value], 'unit':['m','s','kg','J','K']})
planck

## 2. Schwarzschild–Compton crossover

Definitions: the Schwarzschild radius for a classical, nonrotating, uncharged mass is $r_s=2Gm/c^2$; the reduced Compton wavelength is $\bar{\lambda}_C=\hbar/(mc)$. Equate them: $2Gm/c^2=\hbar/(mc)$; multiply by $mc^2$: $2Gm^2=\hbar c$; hence $m=m_P/\sqrt{2}$. Substitution yields both lengths $\sqrt{2}\ell_P$. This order-unity crossover is heuristic; it does not establish a production threshold.

In [ ]:
m = sp.symbols('m', positive=True); Gs, hs, cs = sp.symbols('G hbar c', positive=True)
assert sp.solve(sp.Eq(2*Gs*m/cs**2, hs/(m*cs)), m)[0] == sp.sqrt(hs/(2*Gs*cs))
mp = sp.sqrt(hs*cs/Gs); lp = sp.sqrt(hs*Gs/cs**3)
assert sp.simplify((2*Gs*(mp/sp.sqrt(2))/cs**2) / (sp.sqrt(2)*lp) - 1) == 0
# A grid of chosen masses, not a measured sample.
masses = np.logspace(-8, 2, 500) * mP
r_s = (2*G*masses/c**2).to(u.m)
lam = (hbar/(masses*c)).to(u.m)
fig,ax=plt.subplots(figsize=(8,5)); ax.loglog(masses.value, r_s.value, label=r'$r_s=2Gm/c^2$'); ax.loglog(masses.value, lam.value, label=r'$\bar{\lambda}_C=\hbar/(mc)$')
ax.axvline((mP/np.sqrt(2)).value,color='k',ls=':',label='heuristic crossover')
ax.set(xlabel='Chosen mass (kg)',ylabel='Length (m)',title='Classical horizon and reduced Compton scales (illustration)'); ax.grid(True,which='both',alpha=.25); ax.legend(); plt.show()

## 3. Bekenstein–Hawking entropy: algebra and units

For Schwarzschild, $A=4\pi r_s^2=4\pi(2GM/c^2)^2=16\pi G^2M^2/c^4$. The semiclassical area law is $S_{BH}=k_BA/(4\ell_P^2)$. Using $\ell_P^2=\hbar G/c^3$ gives $S_{BH}=k_BAc^3/(4\hbar G)=4\pi k_BGM^2/(\hbar c)$. The ratio $A/(4\ell_P^2)$ is dimensionless because both numerator and denominator are area; multiplication by $k_B$ restores entropy units. This thermodynamic entropy does not by itself specify experimentally counted microstates.

In [ ]:
M, Gs, cs, hs, kbs = sp.symbols('M G c hbar k_B', positive=True)
rs = 2*Gs*M/cs**2; area = 4*sp.pi*rs**2; lp2 = hs*Gs/cs**3
S_area = kbs*area/(4*lp2); S_mass = 4*sp.pi*kbs*Gs*M**2/(hs*cs)
assert sp.simplify(S_area-S_mass)==0
# Solar-mass example is a derived scale from measured solar mass and constants, not an entropy observation.
Ms = M_sun
area_sun = (4*np.pi*(2*G*Ms/c**2)**2).to(u.m**2)
S_sun = (k_B*area_sun/(4*lP**2)).to(u.J/u.K)
print(f'Schwarzschild area for 1 solar mass: {area_sun:.4g}')
print(f'Semiclassical Bekenstein-Hawking entropy: {S_sun:.4g}')
print('Dimensionless area in Planck units:', (area_sun/(4*lP**2)).decompose())

## 4. EFT suppression, not a UV prediction

The EFT organizes corrections in powers of characteristic energy relative to a cutoff. The displayed $\epsilon^2=(E/E_P)^2$ assumes a unit coefficient solely to visualize dimensional suppression; real terms depend on operators, Wilson coefficients, symmetries, and the actual cutoff. $E/E_P\ll1$ signals a controlled expansion regime; reaching unity signals loss of this simple expansion, not known high-energy behavior. Donoghue's low-energy EFT framework and Wallace's distinction between predictive low-energy theory and unknown UV completion are cited in the companion section.

In [ ]:
fraction = np.logspace(-8, 0.3, 300)
fig,ax=plt.subplots(figsize=(8,5)); ax.loglog(fraction, fraction**2, label=r'illustrative $\epsilon^2$ (unit coefficient)')
ax.axvline(1,color='k',ls=':',label='E = Planck energy'); ax.set(xlabel=r'$E/E_P$',ylabel='Illustrative relative correction',title='EFT power-counting scale (not measured)'); ax.grid(True,which='both',alpha=.25); ax.legend(); plt.show()

## 5. Conditional Lorentz-violating propagation example

Model assumption only: $E^2\simeq p^2c^2[1+\eta(E/E_{QG})^n]$, with $|\eta|(E/E_{QG})^n\ll1$. To leading order, the fractional group-speed shift is $\delta v/c\simeq(n+1)\eta(E/E_{QG})^n/2$. Over a distance $L$, $\Delta t\simeq-(n+1)\eta L(E/E_{QG})^n/(2c)$, up to sign convention. The plotted choices $n=1$, $\eta=1$, $E_{QG}=E_P$, $L=1$ Gpc are illustrative, not fitted data, bounds, or a preferred theory. Real limits require source-lag modeling, cosmological redshift integration, detector response, and a specified operator. Liberati & Maccione review the model dependence; no direct quantum-gravity dispersion measurement is claimed.

In [ ]:
# Explicit toy parameters, not observational inputs or limits.
from astropy.constants import c as c_light
n_toy, eta_toy = 1, 1.0
EQG_toy = EP
L_toy = 1.0 * u.Gpc
Eobs = np.logspace(9, 13, 250) * u.eV
ratio = (Eobs.to(u.J)/EQG_toy).decompose().value
dt = -(n_toy+1)*eta_toy/2 * (L_toy/c_light).to(u.s).value * ratio**n_toy
fig,ax=plt.subplots(figsize=(8,5)); ax.loglog(Eobs.to_value(u.GeV),np.abs(dt),label='absolute toy propagation shift')
ax.set(xlabel='Illustrative observed photon energy (GeV)',ylabel='Absolute illustrative time shift (s)',title='Conditional model curve: no data and no bound'); ax.grid(True,which='both',alpha=.25); ax.legend(); plt.show()
print('Toy parameter choices:', {'n':n_toy,'eta':eta_toy,'E_QG/E_P':(EQG_toy/EP).decompose().value,'distance_Gpc':L_toy.to_value(u.Gpc)})

## 6. Evidence and references

No direct quantum-gravity observation is included. Gravitational-wave, black-hole imaging, and cosmological measurements test classical/semiclassical gravity and constrain some model-dependent departures; they do not select a complete theory. Approach assumptions, strengths, limitations, and references are discussed in [the companion research section](../premium/quantum_gravity.md). The source links include Donoghue (EFT), Wallace (low-energy status), Ashtekar & Bianchi (loop quantum gravity), Polchinski (string theory), Eichhorn (asymptotic safety), Ambjørn et al. (causal dynamical triangulations), and Liberati & Maccione (conditional phenomenology).